# Применение существующих моделей LID на золотом стандарте

Результаты будут проанализированы на двух задачах -- выявление нерелевантных одноязычных текстов и выявление csw в тексте.

Анализируемые методы:
- cld2
- cld3
- fast-langdetect
- lingua
- mediapipe
- fasttext (nllb)
- glotlid
- словарный метод

Методы применялись и анализировались на датасете KazRusCSW-G-T, который мы разметили вручную на уровне текста. Для этого мы отобрали 10 тыс. примеров с обеих платформ (2 telegram-канала и 1 youtube-канал). В процессе разметки встречались комментарии, которые не подходили в силу большого количества орфографических ошибок, неоднозначности языковой принадлежности или чрезмерной грубости -- от них мы избавлялись, что оставило нас с 9700 примерами.

Тегсет включал 4 тега, которые могли комбинироваться друг с другом:
- kz -- комментарий содержит слова казахского языка;
- skz -- комментарий содержит слова казахского языка, написанные с заменой специализированных символов казахского языка;
- ru -- комментарий содержит слова казахского языка;
- unk -- комментарий содержит слова на другом языке (английский, немецкий, якутский и т.д.).

ВАЖНО: блокнот удобнее всего будет запускать в google colab, т.к. меньше проблем с установкой зависимостей (а их много!)

In [1]:
import seaborn as sns
from matplotlib import pyplot as plt
import ast
import pandas as pd
import math

class Experiment:
    def __init__(self, df, method):
        self.df = df
        self.method = method

    def add_probas(self, df_langproba, dropped_cols):
        self.df.drop(columns=df_langproba.columns, errors='ignore', inplace=True)
        self.df = pd.concat([self.df, df_langproba.drop(columns=dropped_cols)], axis=1)
        self.fillna_0()

    def fillna_0(self):
        new_cols = self.df.filter(regex=self.method).select_dtypes(include=['float64']).columns

        self.df[new_cols] = self.df[new_cols].fillna(0)

    def visualize_probas(self, target_col='manual_tag'):
        sns.set()

        fig, axes = plt.subplots(1, 2,figsize=(10,5))
        axes = axes.flatten()

        sns.set(style='whitegrid')

        sns.scatterplot(ax=axes[0], data=self.df, x=f'{self.method}__kaz', y=f'{self.method}__rus', hue=target_col)

        sns.kdeplot(ax=axes[1], data=self.df[[f'{self.method}__kaz', f'{self.method}__rus']])

        fig.suptitle(self.method)

    def save_experiment(self, path):
        self.df.to_csv(path, index=False)

def visualize_tags(df, method, manual_col='manual_tag', x_lang='kaz', y_lang='rus'):
    sns.set()

    nrows = math.ceil(len(df[manual_col].unique())/2)

    fig, axes = plt.subplots(nrows, 2, sharex=True, sharey=True,figsize=(10,nrows*5))
    axes = axes.flatten()

    sns.set(style='whitegrid')

    for i, tag in enumerate(df[manual_col].unique()):
        sns.scatterplot(ax=axes[i], data=df[df[manual_col]==tag], x=f'{method}__{x_lang}', y=f'{method}__{y_lang}')
        axes[i].set_title(tag)

    fig.suptitle(f'Распределение тегов {x_lang}, {y_lang}, полученных с помощью {method}')

In [2]:
def sort_labels(lst):
    order = {
        'lat_kz': 0,
        'kz': 1,
        'skz': 2,
        'ru': 3,
        'en': 4,
        'unk': 5,
    }
    return sorted(lst, key=order.get)

In [3]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix

from sklearn.metrics import classification_report, accuracy_score, precision_recall_curve
from sklearn.metrics import f1_score

import pandas as pd
import numpy as np

In [4]:
sheet_id = "1pIV0z6GWbt_xTweBXHIUXmU4K-84XiTseFbobv7K_xE"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(url, usecols=['text', 'manual_tag'])
df.head()

,text,manual_tag
0,“Сандж” тәуелсіз зерттеу орталығының хабарлауы...,kz
1,Заңгер Алексей Сонның консультациясына мына сі...,kz
2,Шымкентте елуге жуық әйел әкімдік алдына жинал...,kz
3,Өте маңызды жобаға сұхбат кейіпкері ретінде кі...,kz
4,Айсұлтанның “құпиясы” Британиялық Financial ti...,kz-unk


In [5]:
# выкидываем маски
df['text'] = df['text'].str.replace(r'\[[A-Z]+\]', '', regex=True)

In [6]:
save_to_path = 'filtering_experiments.csv'

### Compact Language Detector 2
Репозиторий: https://github.com/CLD2Owners/cld2

Поддерживает 83 языка. Модель представляет собой Naïve Bayesian classifier на основе n-грамм: для казахского и русского предсказания делаются на основе квадграмм(4-грамм). В первую очередь cld2 приводит текст к нижнему регистру, удаляет цифры, пунктуацию и HTML-теги.

#### Обработка

In [7]:
!pip install pyicu pycld2 morfessor

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.2/268.2 kB 6.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 74.0 MB/s eta 0:00:00
  Created wheel for pyicu: filename=pyicu-2.16.2-cp312-cp312-linux_x86_64.whl size=2720233 sha256=2e5b478bd5b055edd3935ace57b493d331a6a523781138a086f6702d7e35bfba
  Stored in directory: /root/.cache/pip/wheels/25/f3/cd/4923c874cedf8cdb8608035f48bb726fa040a98a66e2b13cea
Successfully built pyicu


In [8]:
import pandas as pd

METHOD = 'cld2'
exprmnt = Experiment(df=df, method=METHOD)
exprmnt.df.head()

,text,manual_tag
0,“Сандж” тәуелсіз зерттеу орталығының хабарлауы...,kz
1,Заңгер Алексей Сонның консультациясына мына сі...,kz
2,Шымкентте елуге жуық әйел әкімдік алдына жинал...,kz
3,Өте маңызды жобаға сұхбат кейіпкері ретінде кі...,kz
4,Айсұлтанның “құпиясы” Британиялық Financial ti...,kz-unk


In [15]:
import pycld2 as cld2

lang_dict = {
    'kk': 'kz',
    'ru': 'ru',
}

def cld2__sentence_level(text):
    """
    CLD2 определяет три наиболее вероятных языка.

    Возвращает три объекта:
    - строку из найденных языков, которые расположены в строго определенном порядке kz/skz -> ru -> unk
    - тег языка, который определен как более вероятный
    - вероятности kz, ru, unk
    """
    isReliable, textBytesFound, details = cld2.detect(text)
    dct_all = {l[1]: l[2]/100 for l in details}

    best_lang = details[0][1]

    dct = dict()
    for l in ('kk', 'ru'):
        dct[l] = dct_all.get(l, 0)

    res = []
    # print(details)
    for d in details:
        if d[2]>0:
            if lang_dict.get(d[1]):
                res.append(lang_dict[d[1]])
            else:
                # print(d)
                res.append('unk')

    if not res:
        res = ['unk']

    return '-'.join(sort_labels(set(res))), best_lang, dct

s = 'Білім саласына қатысты сұрақтар бойынша жедел желі нөмірлері. Номера горячей линии по вопросам образования'
cld2__sentence_level(text=s)

('kz-ru', 'kk', {'kk': 0.5, 'ru': 0.49})

In [13]:
df_langproba = pd.DataFrame(index=exprmnt.df.index)

df_langproba[[f'{METHOD}__all_langs', f'{METHOD}__best_lang', f'{METHOD}__probas']] = exprmnt.df['text'].apply(cld2__sentence_level).tolist()
df_langproba = pd.concat([df_langproba, df_langproba[f'{METHOD}__probas'].apply(pd.Series)], axis=1)
df_langproba.rename(columns={
    'kk': f'{METHOD}__kaz',
    'ru': f'{METHOD}__rus',
}, inplace=True)
df_langproba[f'{METHOD}__best_lang'].replace({'kk': 'kz'}, inplace=True)

/tmp/ipykernel_2463/4103955081.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_langproba[f'{METHOD}__best_lang'].replace({'kk': 'kz'}, inplace=True)


In [14]:
df_langproba

,cld2__all_langs,cld2__best_lang,cld2__probas
0,kz,kz,"{'kk': 0.99, 'un': 0.0}"
1,kz,kz,"{'kk': 0.99, 'un': 0.0}"
2,kz,kz,"{'kk': 0.99, 'un': 0.0}"
3,kz,kz,"{'kk': 0.99, 'un': 0.0}"
4,kz,kz,"{'kk': 0.98, 'un': 0.0}"
...,...,...,...
9694,kz,kz,"{'kk': 0.99, 'un': 0.0}"
9695,kz,kz,"{'kk': 0.99, 'un': 0.0}"
9696,kz,kz,"{'kk': 0.99, 'un': 0.0}"
9697,unk,un,{'un': 0.0}


In [ ]:
exprmnt.add_probas(df_langproba, dropped_cols=[f'{METHOD}__probas'])

In [ ]:
exprmnt.df[f'{METHOD}__all_langs'].value_counts()

,count
cld2__all_langs,
ru,4892
kz,3498
unk,1003
ru-unk,143
kz-unk,111
kz-ru,49
kz-ru-unk,3


In [ ]:
exprmnt.df[[f'{METHOD}__kaz', f'{METHOD}__rus']].describe()

,cld2__kaz,cld2__rus
count,9699.000000,9699.000000
mean,0.362215,0.490162
std,0.468461,0.476291
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.590000
75%,0.980000,0.980000
max,0.990000,0.990000


In [ ]:
exprmnt.df['cld2__best_lang'].value_counts()

,count
cld2__best_lang,
ru,4984
kz,3621
un,848
sr,102
ky,88
uz,19
uk,12
tk,8
bg,5


In [ ]:
exprmnt.save_experiment(path=save_to_path)

In [ ]:
df = exprmnt.df

### Compact Language Detector 3
Репозиторий: https://github.com/google/cld3

cld3 разработан для Chrome-браузера, на сегодняшний день существует как публичный архив.

#### Обработка

In [ ]:
METHOD = 'cld3'
exprmnt = Experiment(df=df, method=METHOD)
exprmnt.df.head()

,text,manual_tag,cld2__all_langs,cld2__best_lang,cld2__kaz,cld2__rus
0,“Сандж” тәуелсіз зерттеу орталығының хабарлауы...,kz,kz,kz,0.99,0.0
1,Заңгер Алексей Сонның консультациясына мына сі...,kz,kz-unk,kz,0.98,0.0
2,Шымкентте елуге жуық әйел әкімдік алдына жинал...,kz,kz,kz,0.99,0.0
3,Өте маңызды жобаға сұхбат кейіпкері ретінде кі...,kz,kz,kz,0.98,0.0
4,Айсұлтанның “құпиясы” Британиялық Financial ti...,kz-unk,kz,kz,0.98,0.0


In [ ]:
!sudo apt-get install -y --no-install-recommends g++ protobuf-compiler libprotobuf-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
g++ is already the newest version (4:11.2.0-1ubuntu1).
libprotobuf-dev is already the newest version (3.12.4-1ubuntu7.22.04.6).
protobuf-compiler is already the newest version (3.12.4-1ubuntu7.22.04.6).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [ ]:
!pip install gcld3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 9.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gcld3: filename=gcld3-3.0.13-cp312-cp312-linux_x86_64.whl size=4002758 sha256=51a0303670cdc236d134e646de98910edee2d1cbd708957880b3470e99a43647
  Stored in directory: /root/.cache/pip/wheels/8e/3c/60/7396eb26af0ae770fe9a54d3c4fcb7b54e03919827c488db87
Successfully built gcld3


In [ ]:
import gcld3

detector = gcld3.NNetLanguageIdentifier(min_num_bytes=0, max_num_bytes=1000)

def cld3__sentence_level(text):
    """
    Возвращает два объекта:
    - тег языка, который определен как более вероятный
    - вероятности kz, ru
    """
    res = detector.FindTopNMostFreqLangs(text=text, num_langs=100)
    dct_all = {l.language: l.probability for l in res}

    best_lang = res[0].language

    dct = dict()
    for l in ('kk', 'ru'):
        prob = dct_all.get(l)
        if prob:
            dct[l] = prob
    return best_lang, dct

s = 'Білім саласына қатысты сұрақтар бойынша жедел желі нөмірлері. Номера горячей линии по вопросам образования'
cld3__sentence_level(text=s)

('kk', {'kk': 0.9742560982704163})

In [ ]:
df_langproba = pd.DataFrame(index=exprmnt.df.index)

df_langproba[[f'{METHOD}__best_lang', f'{METHOD}__probas']] = exprmnt.df['text'].apply(cld3__sentence_level).tolist()
df_langproba = pd.concat([df_langproba, df_langproba[f'{METHOD}__probas'].apply(pd.Series)], axis=1)
df_langproba.rename(columns={
    'kk': f'{METHOD}__kaz',
    'ru': f'{METHOD}__rus',
}, inplace=True)

In [ ]:
exprmnt.add_probas(df_langproba, dropped_cols=[f'{METHOD}__probas'])

In [ ]:
exprmnt.df[[f'{METHOD}__kaz', f'{METHOD}__rus']].describe()

,cld3__kaz,cld3__rus
count,9699.000000,9699.000000
mean,0.362240,0.562675
std,0.478067,0.486389
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.974706
75%,0.999937,0.999922
max,1.000000,1.000000


In [ ]:
exprmnt.df['cld3__best_lang'].value_counts()

,count
cld3__best_lang,
ru,5545
kk,3529
ky,279
bg,87
uk,77
be,66
sr,52
mn,22
mk,16


In [ ]:
exprmnt.save_experiment(path=save_to_path)

In [ ]:
df = exprmnt.df

### LangID

https://github.com/saffsd/langid.py (новая версия: https://github.com/adbar/py3langid)

Модель возвращает логиты.

#### Обработка

In [ ]:
!pip3 install py3langid

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 746.1/746.1 kB 10.9 MB/s eta 0:00:00


In [ ]:
METHOD = 'langid'
exprmnt = Experiment(df=df, method=METHOD)
exprmnt.df.head()

,text,manual_tag,cld2__all_langs,cld2__best_lang,cld2__kaz,cld2__rus,cld3__best_lang,cld3__kaz,cld3__rus
0,“Сандж” тәуелсіз зерттеу орталығының хабарлауы...,kz,kz,kz,0.99,0.0,kk,0.999996,0.0
1,Заңгер Алексей Сонның консультациясына мына сі...,kz,kz-unk,kz,0.98,0.0,kk,0.999802,0.0
2,Шымкентте елуге жуық әйел әкімдік алдына жинал...,kz,kz,kz,0.99,0.0,kk,0.999981,0.0
3,Өте маңызды жобаға сұхбат кейіпкері ретінде кі...,kz,kz,kz,0.98,0.0,kk,0.999996,0.0
4,Айсұлтанның “құпиясы” Британиялық Financial ti...,kz-unk,kz,kz,0.98,0.0,kk,0.999997,0.0


In [ ]:
import py3langid as langid

text = 'This text is in English.'

# предсказать наиболее вероятный язык
langid.classify(text)

('en', np.float32(-56.77429))

In [ ]:
# получить сортированный список языков
res = langid.rank(text)
res[:5]

[('en', np.float32(-56.77429)),
 ('la', np.float32(-73.15924)),
 ('af', np.float32(-77.42682)),
 ('br', np.float32(-77.45833)),
 ('nl', np.float32(-80.20705))]

In [ ]:
from py3langid.langid import LanguageIdentifier, MODEL_FILE

identifier = LanguageIdentifier.from_pickled_model(MODEL_FILE, norm_probs=True)

In [ ]:
def langid__sentence_level(text):
    res = langid.rank(text)
    dct = dict()

    for l, logit in res:
        if l in ('kk', 'ru'):
            dct[l] = logit

    if res[0][1] == 0:
        best_lang = 'unk'
    else:
        best_lang = res[0][0]

    return best_lang, dct

df_langproba = pd.DataFrame(index=exprmnt.df.index)

df_langproba[[f'{METHOD}__best_lang', f'{METHOD}__probas']] = exprmnt.df['text'].apply(langid__sentence_level).apply(pd.Series)
df_langproba = pd.concat([df_langproba, df_langproba[f'{METHOD}__probas'].apply(pd.Series)], axis=1)
df_langproba.rename(columns={
    'kk': f'{METHOD}__kaz',
    'ru': f'{METHOD}__rus',
}, inplace=True)

In [ ]:
exprmnt.add_probas(df_langproba, dropped_cols=[f'{METHOD}__probas'])

In [ ]:
exprmnt.save_experiment(path=save_to_path)

In [ ]:
exprmnt.df['langid__best_lang'].value_counts()

,count
langid__best_lang,
ru,5625
kk,3536
bg,197
uk,122
sr,82
mk,73
be,29
mn,18
ky,15


In [ ]:
df = exprmnt.df

### Lingua
Репозиторий: https://pypi.org/project/lingua-language-detector/

Модель допускает многоязычность текста и может разбивать текст на участки по языкам, на которых они написаны.

#### Обработка

In [ ]:
METHOD = 'lingua'

exprmnt = Experiment(df=df, method=METHOD)
exprmnt.df.head()

,text,manual_tag,cld2__all_langs,cld2__best_lang,cld2__kaz,cld2__rus,cld3__best_lang,cld3__kaz,cld3__rus,langid__best_lang,langid__kaz,langid__rus
0,“Сандж” тәуелсіз зерттеу орталығының хабарлауы...,kz,kz,kz,0.99,0.0,kk,0.999996,0.0,kk,-6629.133789,-7293.619141
1,Заңгер Алексей Сонның консультациясына мына сі...,kz,kz-unk,kz,0.98,0.0,kk,0.999802,0.0,kk,-9796.718750,-10485.810547
2,Шымкентте елуге жуық әйел әкімдік алдына жинал...,kz,kz,kz,0.99,0.0,kk,0.999981,0.0,kk,-2515.423340,-2864.809570
3,Өте маңызды жобаға сұхбат кейіпкері ретінде кі...,kz,kz,kz,0.98,0.0,kk,0.999996,0.0,kk,-3253.132324,-3682.371826
4,Айсұлтанның “құпиясы” Британиялық Financial ti...,kz-unk,kz,kz,0.98,0.0,kk,0.999997,0.0,kk,-9981.612305,-10986.610352


In [ ]:
!pip install lingua-language-detector

In [ ]:
from lingua import Language, LanguageDetectorBuilder

languages = [Language.KAZAKH, Language.RUSSIAN, Language.ENGLISH]
# detector = LanguageDetectorBuilder.from_languages(*languages).build()
detector = LanguageDetectorBuilder.from_all_languages().with_preloaded_language_models().build()

def test_lingua(text):
    res = detector.compute_language_confidence_values(text)
    dct = {l.language.name: l.value for l in res}

    if res[0].value == 0:
        best_lang = 'unk'
        dct[best_lang] = 1.0
    else:
        best_lang = res[0].language.name
    return best_lang, dct

s = 'Білім саласына қатысты сұрақтар бойынша жедел желі нөмірлері. Номера горячей линии по вопросам образования'
test_lingua(text=s)

('KAZAKH',
 {'KAZAKH': 1.0,
  'AFRIKAANS': 0.0,
  'ALBANIAN': 0.0,
  'ARABIC': 0.0,
  'ARMENIAN': 0.0,
  'AZERBAIJANI': 0.0,
  'BASQUE': 0.0,
  'BELARUSIAN': 0.0,
  'BENGALI': 0.0,
  'BOKMAL': 0.0,
  'BOSNIAN': 0.0,
  'BULGARIAN': 0.0,
  'CATALAN': 0.0,
  'CHINESE': 0.0,
  'CROATIAN': 0.0,
  'CZECH': 0.0,
  'DANISH': 0.0,
  'DUTCH': 0.0,
  'ENGLISH': 0.0,
  'ESPERANTO': 0.0,
  'ESTONIAN': 0.0,
  'FINNISH': 0.0,
  'FRENCH': 0.0,
  'GANDA': 0.0,
  'GEORGIAN': 0.0,
  'GERMAN': 0.0,
  'GREEK': 0.0,
  'GUJARATI': 0.0,
  'HEBREW': 0.0,
  'HINDI': 0.0,
  'HUNGARIAN': 0.0,
  'ICELANDIC': 0.0,
  'INDONESIAN': 0.0,
  'IRISH': 0.0,
  'ITALIAN': 0.0,
  'JAPANESE': 0.0,
  'KOREAN': 0.0,
  'LATIN': 0.0,
  'LATVIAN': 0.0,
  'LITHUANIAN': 0.0,
  'MACEDONIAN': 0.0,
  'MALAY': 0.0,
  'MAORI': 0.0,
  'MARATHI': 0.0,
  'MONGOLIAN': 0.0,
  'NYNORSK': 0.0,
  'PERSIAN': 0.0,
  'POLISH': 0.0,
  'PORTUGUESE': 0.0,
  'PUNJABI': 0.0,
  'ROMANIAN': 0.0,
  'RUSSIAN': 0.0,
  'SERBIAN': 0.0,
  'SHONA': 0.0,
  'SLOVA

In [ ]:
detector = LanguageDetectorBuilder.from_all_languages().with_preloaded_language_models().build()

def lingua__sentence_level(res):
    """
    Функция принимает результат работы модели (вероятности), полученный в результате параллельных вычислений.
    Возвращает наиболее вероятный язык и вероятности для 'KAZAKH', 'RUSSIAN'
    """
    # res = detector.compute_language_confidence_values(text)
    if type(res)!=list:
        return 'unk', None
    dct = dict()

    if res[0].value == 0:
        best_lang = 'unk'
        # dct[best_lang] = 1.0
    else:
        for l in res:
            if l.language.name in ('KAZAKH', 'RUSSIAN'):
                dct[l.language.name] = l.value
        best_lang = res[0].language.name
    return best_lang, dct

In [ ]:
lang_dict = {
    'KAZAKH': 'kz',
    'RUSSIAN': 'ru',
}

def get_lingua_labels(lst):
    """
    Принимает список вероятных языков, сортирует в определенном выше порядке и объединяет в одну строку (см. cld2__sentence_level)
    """
    res = []
    for l in lst:
        # print(l.language.name)
        if lang_dict.get(l.language.name):
            res.append(lang_dict[l.language.name])
        else:
            res.append('unk')

    if not res:
        res = ['unk']
    return '-'.join(sort_labels(set(res)))

In [ ]:
df_langproba = pd.DataFrame(index=exprmnt.df.index)

df_langproba[f'{METHOD}__res'] = pd.Series(detector.compute_language_confidence_values_in_parallel(exprmnt.df['text']))
df_langproba[[f'{METHOD}__best_lang', f'{METHOD}__probas']] = df_langproba[f'{METHOD}__res'].apply(lingua__sentence_level).apply(pd.Series)
df_langproba = pd.concat([df_langproba, df_langproba[f'{METHOD}__probas'].apply(pd.Series)], axis=1)
df_langproba.rename(columns={
    'KAZAKH': f'{METHOD}__kaz',
    'RUSSIAN': f'{METHOD}__rus',
}, inplace=True)

df_langproba[f'{METHOD}__langs'] = pd.Series(detector.detect_multiple_languages_in_parallel_of(exprmnt.df['text'])) # участки текста, размеченные по языкам
df_langproba[f'{METHOD}__all_langs'] = df_langproba[f'{METHOD}__langs'].apply(get_lingua_labels)

In [ ]:
df_langproba[f'{METHOD}__best_lang'].value_counts()/len(df_langproba)

,count
lingua__best_lang,
RUSSIAN,0.596247
KAZAKH,0.388597
MONGOLIAN,0.004021
UKRAINIAN,0.003918
BULGARIAN,0.002681
MACEDONIAN,0.001547
SERBIAN,0.001031
BELARUSIAN,0.001031
ENGLISH,0.000412


In [ ]:
df_langproba[f'{METHOD}__all_langs'].value_counts()

,count
lingua__all_langs,
kz,3196
ru,2732
ru-unk,2529
unk,339
kz-ru,326
kz-unk,316
kz-ru-unk,261


In [ ]:
exprmnt.add_probas(df_langproba, dropped_cols=[f'{METHOD}__probas', f'{METHOD}__res'])

In [ ]:
exprmnt.save_experiment(path=save_to_path)

In [ ]:
df = exprmnt.df

### MediaPipe (Language deteciton)

https://ai.google.dev/edge/mediapipe/solutions/text/language_detector


архитектура: https://storage.googleapis.com/mediapipe-assets/LanguageDetector%20Model%20Card.pdf

#### Обработка

In [ ]:
!pip install -q mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 9.3 MB/s eta 0:00:00


In [ ]:
!wget -O detector.tflite -q https://storage.googleapis.com/mediapipe-models/language_detector/language_detector/float32/latest/language_detector.tflite

In [ ]:
import pandas as pd

METHOD = 'mediapipe'
exprmnt = Experiment(df=df, method=METHOD)
exprmnt.df.head()

,text,manual_tag,cld2__all_langs,cld2__best_lang,cld2__kaz,cld2__rus,cld3__best_lang,cld3__kaz,cld3__rus,langid__best_lang,langid__kaz,langid__rus,lingua__best_lang,lingua__kaz,lingua__rus,lingua__langs,lingua__all_langs
0,“Сандж” тәуелсіз зерттеу орталығының хабарлауы...,kz,kz,kz,0.99,0.0,kk,0.999996,0.0,kk,-6629.133789,-7293.619141,KAZAKH,1.0,0.0,"[(0, 565, 76, KAZAKH)]",kz
1,Заңгер Алексей Сонның консультациясына мына сі...,kz,kz-unk,kz,0.98,0.0,kk,0.999802,0.0,kk,-9796.718750,-10485.810547,KAZAKH,1.0,0.0,"[(0, 789, 96, KAZAKH)]",kz
2,Шымкентте елуге жуық әйел әкімдік алдына жинал...,kz,kz,kz,0.99,0.0,kk,0.999981,0.0,kk,-2515.423340,-2864.809570,KAZAKH,1.0,0.0,"[(0, 237, 30, KAZAKH)]",kz
3,Өте маңызды жобаға сұхбат кейіпкері ретінде кі...,kz,kz,kz,0.98,0.0,kk,0.999996,0.0,kk,-3253.132324,-3682.371826,KAZAKH,1.0,0.0,"[(0, 297, 39, KAZAKH)]",kz
4,Айсұлтанның “құпиясы” Британиялық Financial ti...,kz-unk,kz,kz,0.98,0.0,kk,0.999997,0.0,kk,-9981.612305,-10986.610352,KAZAKH,1.0,0.0,"[(0, 268, 32, KAZAKH), (268, 287, 2, RUSSIAN),...",kz-ru


In [ ]:
from mediapipe.tasks import python
from mediapipe.tasks.python import text as mptext

In [ ]:
base_options = python.BaseOptions(model_asset_path="detector.tflite")

options = mptext.LanguageDetectorOptions(base_options=base_options, score_threshold=1e-8)
detector = mptext.LanguageDetector.create_from_options(options)

def mediapipe__sentence_level(text, detector):
    """
    Возвращает наиболее вероятный язык и вероятности для kk и ru
    """
    res = detector.detect(text=text)
    dct_all = {l.language_code: l.probability for l in res.detections}

    best_lang = res.detections[0].language_code

    dct = dict()
    for l in ('kk', 'ru'):
        prob = dct_all.get(l)
        if prob:
            dct[l] = prob
    return best_lang, dct

s = 'Білім саласына қатысты сұрақтар бойынша жедел желі нөмірлері. Номера горячей линии по вопросам образования'
mediapipe__sentence_level(text=s, detector=detector)

('kk', {'kk': 0.7249384522438049, 'ru': 0.007838883437216282})

In [ ]:
df_langproba = pd.DataFrame(index=exprmnt.df.index)

df_langproba[[f'{METHOD}__best_lang', f'{METHOD}__probas']] = exprmnt.df['text'].apply(mediapipe__sentence_level, detector=detector).tolist()
df_langproba = pd.concat([df_langproba, df_langproba[f'{METHOD}__probas'].apply(pd.Series)], axis=1)
df_langproba.rename(columns={
    'kk': f'{METHOD}__kaz',
    'ru': f'{METHOD}__rus',
}, inplace=True)

In [ ]:
exprmnt.add_probas(df_langproba, dropped_cols=['mediapipe__probas'])

In [ ]:
exprmnt.save_experiment(path=save_to_path)

In [ ]:
df = exprmnt.df

## Fasttext-based

Обязательно сохранить прогресс перед переустановкой numpy!
После перезапуска ядра запустить подготовительные ячейки в самом начале блокнота

In [ ]:
!pip install numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 80.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
py3langid 0.3.0 requires numpy>=2.0.0; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 whic

In [ ]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.4-py3-none-any.whl (314 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653905 sha256=749ccf2d18079acff1a4abfd4773620abe73dc92ad8475a555817981c042d723
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
df = pd.read_csv(save_to_path)
df.head()

,text,manual_tag,cld2__all_langs,cld2__best_lang,cld2__kaz,cld2__rus,cld3__best_lang,cld3__kaz,cld3__rus,langid__best_lang,langid__kaz,langid__rus,lingua__best_lang,lingua__kaz,lingua__rus,lingua__langs,lingua__all_langs,mediapipe__best_lang,mediapipe__kaz,mediapipe__rus
0,“Сандж” тәуелсіз зерттеу орталығының хабарлауы...,kz,kz,kz,0.99,0.0,kk,0.999996,0.0,kk,-6629.1340,-7293.6190,KAZAKH,1.0,0.0,"[DetectionResult(start_index=0, end_index=565,...",kz,kk,0.999208,1.311968e-06
1,Заңгер Алексей Сонның консультациясына мына сі...,kz,kz-unk,kz,0.98,0.0,kk,0.999802,0.0,kk,-9796.7190,-10485.8110,KAZAKH,1.0,0.0,"[DetectionResult(start_index=0, end_index=789,...",kz,kk,0.998663,1.822960e-06
2,Шымкентте елуге жуық әйел әкімдік алдына жинал...,kz,kz,kz,0.99,0.0,kk,0.999981,0.0,kk,-2515.4233,-2864.8096,KAZAKH,1.0,0.0,"[DetectionResult(start_index=0, end_index=237,...",kz,kk,0.999861,1.862460e-08
3,Өте маңызды жобаға сұхбат кейіпкері ретінде кі...,kz,kz,kz,0.98,0.0,kk,0.999996,0.0,kk,-3253.1323,-3682.3718,KAZAKH,1.0,0.0,"[DetectionResult(start_index=0, end_index=297,...",kz,kk,0.999542,3.649036e-08
4,Айсұлтанның “құпиясы” Британиялық Financial ti...,kz-unk,kz,kz,0.98,0.0,kk,0.999997,0.0,kk,-9981.6120,-10986.6100,KAZAKH,1.0,0.0,"[DetectionResult(start_index=0, end_index=268,...",kz-ru,kk,0.993801,7.061346e-06


### Fast-langdetect

https://github.com/LlmKira/fast-langdetect

Обертка, ускоряющая работу FastText

#### Обработка

In [ ]:
!pip install fast-langdetect

In [ ]:
import pandas as pd

METHOD = 'fast_langdetect'
exprmnt = Experiment(df=df, method=METHOD)
exprmnt.df.head()

,text,manual_tag,cld2__all_langs,cld2__best_lang,cld2__kaz,cld2__rus,cld3__best_lang,cld3__kaz,cld3__rus,langid__best_lang,langid__kaz,langid__rus,lingua__best_lang,lingua__kaz,lingua__rus,lingua__langs,lingua__all_langs,mediapipe__best_lang,mediapipe__kaz,mediapipe__rus
0,“Сандж” тәуелсіз зерттеу орталығының хабарлауы...,kz,kz,kz,0.99,0.0,kk,0.999996,0.0,kk,-6629.1340,-7293.6190,KAZAKH,1.0,0.0,"[DetectionResult(start_index=0, end_index=565,...",kz,kk,0.999208,1.311968e-06
1,Заңгер Алексей Сонның консультациясына мына сі...,kz,kz-unk,kz,0.98,0.0,kk,0.999802,0.0,kk,-9796.7190,-10485.8110,KAZAKH,1.0,0.0,"[DetectionResult(start_index=0, end_index=789,...",kz,kk,0.998663,1.822960e-06
2,Шымкентте елуге жуық әйел әкімдік алдына жинал...,kz,kz,kz,0.99,0.0,kk,0.999981,0.0,kk,-2515.4233,-2864.8096,KAZAKH,1.0,0.0,"[DetectionResult(start_index=0, end_index=237,...",kz,kk,0.999861,1.862460e-08
3,Өте маңызды жобаға сұхбат кейіпкері ретінде кі...,kz,kz,kz,0.98,0.0,kk,0.999996,0.0,kk,-3253.1323,-3682.3718,KAZAKH,1.0,0.0,"[DetectionResult(start_index=0, end_index=297,...",kz,kk,0.999542,3.649036e-08
4,Айсұлтанның “құпиясы” Британиялық Financial ti...,kz-unk,kz,kz,0.98,0.0,kk,0.999997,0.0,kk,-9981.6120,-10986.6100,KAZAKH,1.0,0.0,"[DetectionResult(start_index=0, end_index=268,...",kz-ru,kk,0.993801,7.061346e-06


In [ ]:
from fast_langdetect import detect, LangDetector, LangDetectConfig

def fast_langdetect__sentence_level(text):
    """
    Модель считает вероятности для всех поддерживаемых языков. Нас интересуют только вероятности для казахского и русского
    """
    res = detect(text, model='auto', k=-1)
    dct_all = {l['lang']: l['score'] for l in res}

    best_lang = res[0]['lang']

    dct = dict()
    for l in ('kk', 'ru'):
        prob = dct_all.get(l)
        if prob:
            dct[l] = prob
    return best_lang, dct

s = 'Білім саласына қатысты сұрақтар бойынша жедел желі нөмірлері. Номера горячей линии по вопросам образования'
fast_langdetect__sentence_level(s)

('kk', {'kk': 0.996268093585968, 'ru': 0.00010874634608626366})

In [ ]:
df_langproba = pd.DataFrame(index=exprmnt.df.index)

df_langproba[[f'{METHOD}__best_lang', f'{METHOD}__probas']] = exprmnt.df['text'].apply(fast_langdetect__sentence_level).tolist()
df_langproba = pd.concat([df_langproba, df_langproba[f'{METHOD}__probas'].apply(pd.Series)], axis=1)
df_langproba.rename(columns={
    'kk': f'{METHOD}__kaz',
    'ru': f'{METHOD}__rus',
}, inplace=True)

In [ ]:
exprmnt.add_probas(df_langproba, dropped_cols=[f'{METHOD}__probas'])

In [ ]:
exprmnt.df[f'{METHOD}__best_lang'].value_counts()

,count
fast_langdetect__best_lang,
ru,5977
kk,3496
ky,128
uk,33
tt,13
bg,10
mn,7
sah,7
sr,6


In [ ]:
exprmnt.df[[f'{METHOD}__kaz', f'{METHOD}__rus']].describe()

,fast_langdetect__kaz,fast_langdetect__rus
count,9699.000000,9699.000000
mean,0.353513,0.602095
std,0.467808,0.471251
min,0.000000,0.000000
25%,0.000000,0.000171
50%,0.000166,0.975215
75%,0.994589,0.997384
max,1.000000,1.000000


In [ ]:
exprmnt.save_experiment(path=save_to_path)

In [ ]:
df = exprmnt.df

### GlotLID

In [ ]:
import pandas as pd

METHOD = 'glotlid'
exprmnt = Experiment(df=df, method=METHOD)
exprmnt.df.head()

,text,manual_tag,cld2__all_langs,cld2__best_lang,cld2__kaz,cld2__rus,cld3__best_lang,cld3__kaz,cld3__rus,langid__best_lang,...,lingua__kaz,lingua__rus,lingua__langs,lingua__all_langs,mediapipe__best_lang,mediapipe__kaz,mediapipe__rus,fast_langdetect__best_lang,fast_langdetect__kaz,fast_langdetect__rus
0,“Сандж” тәуелсіз зерттеу орталығының хабарлауы...,kz,kz,kz,0.99,0.0,kk,0.999996,0.0,kk,...,1.0,0.0,"[DetectionResult(start_index=0, end_index=565,...",kz,kk,0.999208,1.311968e-06,kk,0.998260,0.000022
1,Заңгер Алексей Сонның консультациясына мына сі...,kz,kz-unk,kz,0.98,0.0,kk,0.999802,0.0,kk,...,1.0,0.0,"[DetectionResult(start_index=0, end_index=789,...",kz,kk,0.998663,1.822960e-06,kk,0.816011,0.052039
2,Шымкентте елуге жуық әйел әкімдік алдына жинал...,kz,kz,kz,0.99,0.0,kk,0.999981,0.0,kk,...,1.0,0.0,"[DetectionResult(start_index=0, end_index=237,...",kz,kk,0.999861,1.862460e-08,kk,0.999802,0.000114
3,Өте маңызды жобаға сұхбат кейіпкері ретінде кі...,kz,kz,kz,0.98,0.0,kk,0.999996,0.0,kk,...,1.0,0.0,"[DetectionResult(start_index=0, end_index=297,...",kz,kk,0.999542,3.649036e-08,kk,1.000000,0.000000
4,Айсұлтанның “құпиясы” Британиялық Financial ti...,kz-unk,kz,kz,0.98,0.0,kk,0.999997,0.0,kk,...,1.0,0.0,"[DetectionResult(start_index=0, end_index=268,...",kz-ru,kk,0.993801,7.061346e-06,kk,0.997506,0.000012


In [ ]:
import fasttext
from huggingface_hub import hf_hub_download

glotlid_model_path = hf_hub_download(repo_id="cis-lmu/glotlid", filename="model.bin")
glotlid_model = fasttext.load_model(glotlid_model_path)
glotlid_model.predict("коррупция и вляние россии болмаганда бари баскаша болар еді", 3)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.bin:   0%|          | 0.00/1.69G [00:00<?, ?B/s]

(('__label__kaz_Cyrl', '__label__rus_Cyrl', '__label__chu_Cyrl'),
 array([0.82945824, 0.12417033, 0.01198109]))

In [ ]:
def glotlid__sentence_level(text, model):
    """
    Модель считает вероятности для всех поддерживаемых языков.
    Функция возвращает наиболее вероятный язык и распределение вероятностей для казахского и русского.
    """
    # res = model.predict(text, k=3)
    labels, probas = model.predict(text,k=100, threshold=1e-8)
    dct_all = dict(zip(labels, probas))

    best_lang = labels[0]

    dct = dict()
    for l in ('__label__kaz_Cyrl', '__label__rus_Cyrl'):
        prob = dct_all.get(l)
        if prob:
            dct[l.replace('__label', 'glotlid')] = prob
    return best_lang, dct

s = 'мүмкін тайный покупатель сияқты шығарсың'
glotlid__sentence_level(text=s, model=glotlid_model)

('__label__kaz_Cyrl',
 {'glotlid__kaz_Cyrl': 0.9982357621192932,
  'glotlid__rus_Cyrl': 1.1178888598806225e-05})

In [ ]:
df_langproba = pd.DataFrame(index=exprmnt.df.index)

df_langproba[[f'{METHOD}__best_lang', f'{METHOD}__probas']] = exprmnt.df['text'].apply(glotlid__sentence_level, model=glotlid_model).tolist()
df_langproba = pd.concat([df_langproba, df_langproba[f'{METHOD}__probas'].apply(pd.Series)], axis=1)
df_langproba.rename(columns={
    f'{METHOD}__kaz_Cyrl': f'{METHOD}__kaz',
    f'{METHOD}__rus_Cyrl': f'{METHOD}__rus',
}, inplace=True)

In [ ]:
exprmnt.add_probas(df_langproba, dropped_cols=[f'{METHOD}__probas'])

In [ ]:
exprmnt.df[f'{METHOD}__best_lang'].value_counts()

,count
glotlid__best_lang,
__label__rus_Cyrl,5798
__label__kaz_Cyrl,3584
__label__kir_Cyrl,141
__label__nog_Cyrl,24
__label__orv_Cyrl,22
__label__uzn_Cyrl,13
__label__bul_Cyrl,12
__label__ukr_Cyrl,12
__label__sah_Cyrl,11


In [ ]:
exprmnt.df[[f'{METHOD}__kaz',f'{METHOD}__rus']].describe()

,glotlid__kaz,glotlid__rus
count,9699.000000,9699.000000
mean,0.368423,0.590590
std,0.478481,0.482831
min,0.000000,0.000000
25%,0.000011,0.000000
50%,0.000043,0.989969
75%,0.999994,0.999325
max,1.000010,1.000010


In [ ]:
exprmnt.save_experiment(path=save_to_path)

In [ ]:
df = exprmnt.df

### Fasttext

In [ ]:
METHOD = 'fasttext'
exprmnt = Experiment(df=df, method=METHOD)
exprmnt.df.head()

,text,manual_tag,cld2__all_langs,cld2__best_lang,cld2__kaz,cld2__rus,cld3__best_lang,cld3__kaz,cld3__rus,langid__best_lang,...,lingua__all_langs,mediapipe__best_lang,mediapipe__kaz,mediapipe__rus,fast_langdetect__best_lang,fast_langdetect__kaz,fast_langdetect__rus,glotlid__best_lang,glotlid__kaz,glotlid__rus
0,“Сандж” тәуелсіз зерттеу орталығының хабарлауы...,kz,kz,kz,0.99,0.0,kk,0.999996,0.0,kk,...,kz,kk,0.999208,1.311968e-06,kk,0.998260,0.000022,__label__kaz_Cyrl,0.999986,0.0
1,Заңгер Алексей Сонның консультациясына мына сі...,kz,kz-unk,kz,0.98,0.0,kk,0.999802,0.0,kk,...,kz,kk,0.998663,1.822960e-06,kk,0.816011,0.052039,__label__kaz_Cyrl,0.999995,0.0
2,Шымкентте елуге жуық әйел әкімдік алдына жинал...,kz,kz,kz,0.99,0.0,kk,0.999981,0.0,kk,...,kz,kk,0.999861,1.862460e-08,kk,0.999802,0.000114,__label__kaz_Cyrl,1.000010,0.0
3,Өте маңызды жобаға сұхбат кейіпкері ретінде кі...,kz,kz,kz,0.98,0.0,kk,0.999996,0.0,kk,...,kz,kk,0.999542,3.649036e-08,kk,1.000000,0.000000,__label__kaz_Cyrl,1.000009,0.0
4,Айсұлтанның “құпиясы” Британиялық Financial ti...,kz-unk,kz,kz,0.98,0.0,kk,0.999997,0.0,kk,...,kz-ru,kk,0.993801,7.061346e-06,kk,0.997506,0.000012,__label__kaz_Cyrl,1.000001,0.0


In [ ]:
import fasttext
from huggingface_hub import hf_hub_download

fasttext_model_path = hf_hub_download(repo_id="facebook/fasttext-language-identification", filename="model.bin")
fasttext_model = fasttext.load_model(fasttext_model_path)
fasttext_model.predict("коррупция и вляние россии болмаганда бари баскаша болар еді", 3)

model.bin:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

(('__label__kaz_Cyrl', '__label__ukr_Cyrl', '__label__tat_Cyrl'),
 array([0.79145634, 0.14584348, 0.02999006]))

In [ ]:
def fasttext__sentence_level(text, model):
    """
    Модель считает вероятности для всех поддерживаемых языков.
    Функция возвращает наиболее вероятный язык и распределение вероятностей для казахского и русского.
    """
    # res = model.predict(text, k=3)
    labels, probas = model.predict(text,k=-1)
    dct_all = dict(zip(labels, probas))

    best_lang = labels[0]

    dct = dict()
    for l in ('__label__kaz_Cyrl', '__label__rus_Cyrl'):
        prob = dct_all.get(l)
        if prob:
            dct[l.replace('__label', 'fasttext')] = prob
    return best_lang, dct

s = 'мүмкін тайный покупатель сияқты шығарсың'
fasttext__sentence_level(text=s, model=fasttext_model)

('__label__kaz_Cyrl',
 {'fasttext__kaz_Cyrl': 1.000008463859558,
  'fasttext__rus_Cyrl': 1.0069999007100705e-05})

In [ ]:
df_langproba = pd.DataFrame(index=exprmnt.df.index)

df_langproba[[f'{METHOD}__best_lang', f'{METHOD}__probas']] = exprmnt.df['text'].apply(fasttext__sentence_level, model=fasttext_model).tolist()
df_langproba = pd.concat([df_langproba, df_langproba[f'{METHOD}__probas'].apply(pd.Series)], axis=1)
df_langproba.rename(columns={
    f'{METHOD}__kaz_Cyrl': f'{METHOD}__kaz',
    f'{METHOD}__rus_Cyrl': f'{METHOD}__rus',
}, inplace=True)

In [ ]:
exprmnt.add_probas(df_langproba, dropped_cols=[f'{METHOD}__probas'])

In [ ]:
exprmnt.df[f'{METHOD}__best_lang'].value_counts()

,count
fasttext__best_lang,
__label__rus_Cyrl,5778
__label__kaz_Cyrl,3633
__label__kir_Cyrl,140
__label__ukr_Cyrl,35
__label__tat_Cyrl,17
__label__yue_Hant,10
__label__che_Cyrl,9
__label__tgk_Cyrl,7
__label__bul_Cyrl,7


In [ ]:
exprmnt.df[[f'{METHOD}__kaz',f'{METHOD}__rus']].describe()

,fasttext__kaz,fasttext__rus
count,9699.000000,9699.000000
mean,0.374173,0.583141
std,0.479793,0.479172
min,0.000010,0.000010
25%,0.000024,0.000010
50%,0.000225,0.972691
75%,1.000010,0.996376
max,1.000010,1.000010


In [ ]:
exprmnt.save_experiment(path=save_to_path)

In [ ]:
df = exprmnt.df

## Словарный метод

Словарный метод, опирающийся на множество источников:
- спелл-чекер hunspell
- толковый словарь казахского языка
- списки суффиксов казахского языка, составленные на основе онлайн-учебника

In [ ]:
METHOD = 'dict'
exprmnt = Experiment(df=df, method=METHOD)
exprmnt.df.head()

,text,manual_tag,cld2__all_langs,cld2__best_lang,cld2__kaz,cld2__rus,cld3__best_lang,cld3__kaz,cld3__rus,langid__best_lang,...,fast_langdetect__kaz,fast_langdetect__rus,glotlid__best_lang,glotlid__kaz,glotlid__rus,fasttext__best_lang,fasttext__kaz,fasttext__rus,tokens,spans
0,“Сандж” тәуелсіз зерттеу орталығының хабарлауы...,kz,kz,kz,0.99,0.0,kk,0.999996,0.0,kk,...,0.998260,0.000022,__label__kaz_Cyrl,0.999986,0.0,__label__kaz_Cyrl,1.00001,0.00001,"[Сандж, тәуелсіз, зерттеу, орталығының, хабарл...","[(1, 6), (8, 16), (17, 24), (25, 36), (37, 49)..."
1,Заңгер Алексей Сонның консультациясына мына сі...,kz,kz-unk,kz,0.98,0.0,kk,0.999802,0.0,kk,...,0.816011,0.052039,__label__kaz_Cyrl,0.999995,0.0,__label__kaz_Cyrl,1.00001,0.00001,"[Заңгер, Алексей, Сонның, консультациясына, мы...","[(0, 6), (7, 14), (15, 21), (22, 38), (39, 43)..."
2,Шымкентте елуге жуық әйел әкімдік алдына жинал...,kz,kz,kz,0.99,0.0,kk,0.999981,0.0,kk,...,0.999802,0.000114,__label__kaz_Cyrl,1.000010,0.0,__label__kaz_Cyrl,1.00001,0.00001,"[Шымкентте, елуге, жуық, әйел, әкімдік, алдына...","[(0, 9), (10, 15), (16, 20), (21, 25), (26, 33..."
3,Өте маңызды жобаға сұхбат кейіпкері ретінде кі...,kz,kz,kz,0.98,0.0,kk,0.999996,0.0,kk,...,1.000000,0.000000,__label__kaz_Cyrl,1.000009,0.0,__label__kaz_Cyrl,1.00001,0.00001,"[Өте, маңызды, жобаға, сұхбат, кейіпкері, реті...","[(0, 3), (4, 11), (12, 18), (19, 25), (26, 35)..."
4,Айсұлтанның “құпиясы” Британиялық Financial ti...,kz-unk,kz,kz,0.98,0.0,kk,0.999997,0.0,kk,...,0.997506,0.000012,__label__kaz_Cyrl,1.000001,0.0,__label__kaz_Cyrl,1.00001,0.00001,"[Айсұлтанның, құпиясы, Британиялық, Financial,...","[(0, 11), (13, 20), (22, 33), (34, 43), (44, 4..."


#### Сбор словарей

In [ ]:
# скачиваем словари для казахского
!wget https://raw.githubusercontent.com/taem/aspell-kk/refs/heads/master/words

!wget https://raw.githubusercontent.com/kergalym/myspell-kk/refs/heads/upstream/kk_KZ.dic
!wget https://raw.githubusercontent.com/kergalym/myspell-kk/refs/heads/upstream/kk_KZ.aff
!wget https://raw.githubusercontent.com/kergalym/myspell-kk/refs/heads/upstream/kk_noun_adj.aff
!wget https://raw.githubusercontent.com/kergalym/myspell-kk/refs/heads/upstream/kk_noun_adj.dic

--2026-05-14 22:44:12--  https://raw.githubusercontent.com/taem/aspell-kk/refs/heads/master/words
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 867799 (847K) [text/plain]
Saving to: ‘words’

words               100%[===================>] 847.46K  --.-KB/s    in 0.05s   

2026-05-14 22:44:12 (15.5 MB/s) - ‘words’ saved [867799/867799]

--2026-05-14 22:44:12--  https://raw.githubusercontent.com/kergalym/myspell-kk/refs/heads/upstream/kk_KZ.dic
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2130055 (2.0M) [text/plain]
Saving to: ‘

In [ ]:
# словари
with open('kk_KZ.dic', 'r', encoding='utf8') as f:
    kaz_dict1 = list(f.read().lower().splitlines())
    kaz_dict1 = list(map(lambda x: x[:x.find('/')], kaz_dict1))
    print(len(kaz_dict1))
# 54064

with open('kk_noun_adj.dic', 'r', encoding='utf8') as f:
    kaz_dict2 = list(f.read().lower().splitlines())
    kaz_dict2 = list(map(lambda x: x[:x.find('/')], kaz_dict2))
    print(len(kaz_dict2))
# 20854

with open('words', 'r', encoding='utf8') as f:
    kaz_dict3 = list(f.read().lower().splitlines())
    print(len(kaz_dict3))
# 53717

# толковый словарь казахского языка (не опубликован)
big_kz_dict_path = 'big_kz_dict.txt'
with open(big_kz_dict_path, 'r', encoding='utf8') as f:
    kaz_dict4 = list(f.read().lower().splitlines())
    print(len(kaz_dict4))
# 76006

54064
20854
53717
76006


In [ ]:
# объединенный словарь
kaz_dict = set(kaz_dict1 + kaz_dict2 + kaz_dict3 + kaz_dict4)
print(len(kaz_dict))
# 64840

107943


In [ ]:
# устанавливаем спеллчекер
!sudo apt-get update
!sudo apt-get install libhunspell-dev python3-dev
!pip install hunspell

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,602 kB]
Get:14 h

In [ ]:
# объединяем словари для спеллчекера
!echo "$(wc -l < kk_KZ.dic) $(wc -l < kk_noun_adj.dic) $(($(wc -l < kk_KZ.dic) + $(wc -l < kk_noun_adj.dic)))" > merged_dics.dic
!grep -v '^$' kk_KZ.dic >> merged_dics.dic
!grep -v '^$' kk_noun_adj.dic >> merged_dics.dic

!cp kk_KZ.aff merged_dics.aff

In [ ]:
import hunspell

kz_spellcheck = hunspell.HunSpell('merged_dics.dic', 'merged_dics.aff')

correct_word = 'қазақстанға'.encode('utf-8')
incorrect_word = 'бир'.encode('utf-8')

# Check words
print(f'Is {correct_word.decode()} spelled correctly?', kz_spellcheck.spell(correct_word))
print(f'Is {incorrect_word.decode()} spelled correctly?', kz_spellcheck.spell(incorrect_word))

# Suggestions
print('Correct options:', kz_spellcheck.suggest(incorrect_word))

# Stemming (analyze)
res = kz_spellcheck.analyze(correct_word)
print(f'Word  {correct_word.decode()} normalized:', res[0].decode())

Is қазақстанға spelled correctly? True
Is бир spelled correctly? False
Correct options: ['би', 'обир', 'бәр', 'биі', 'бір', 'биң', 'бүр', 'бұр', 'бор', 'бар', 'бит', 'тир', 'бие', 'бер', 'бис']
Word  қазақстанға normalized:  st:қазақстан fl:H


In [ ]:
# скачиваем словари для русского спелл-чекера
!wget -O ru_RU.dic https://raw.githubusercontent.com/wooorm/dictionaries/refs/heads/main/dictionaries/ru/index.dic
!wget -O ru_RU.aff https://raw.githubusercontent.com/wooorm/dictionaries/refs/heads/main/dictionaries/ru/index.aff

--2026-05-14 22:46:18--  https://raw.githubusercontent.com/wooorm/dictionaries/refs/heads/main/dictionaries/ru/index.dic
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3473191 (3.3M) [text/plain]
Saving to: ‘ru_RU.dic’

ru_RU.dic           100%[===================>]   3.31M  --.-KB/s    in 0.1s    

2026-05-14 22:46:18 (33.5 MB/s) - ‘ru_RU.dic’ saved [3473191/3473191]

--2026-05-14 22:46:18--  https://raw.githubusercontent.com/wooorm/dictionaries/refs/heads/main/dictionaries/ru/index.aff
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Le

In [ ]:
ru_spellcheck = hunspell.HunSpell('ru_RU.dic', 'ru_RU.aff')

correct_word = 'остановленного'.encode('utf-8')
incorrect_word = 'остоновочный'.encode('utf-8')

# Check words
print(f'Is {correct_word.decode()} spelled correctly?', ru_spellcheck.spell(correct_word))
print(f'Is {incorrect_word.decode()} spelled correctly?', ru_spellcheck.spell(incorrect_word))

# Suggestions
print('Correct options:', ru_spellcheck.suggest(incorrect_word))

# Stemming (analyze)
res = ru_spellcheck.analyze(correct_word)
print(f'Word  {correct_word.decode()} normalized:', res[0].decode()) # res[0].decode().split()[0].split(':')[1]

Is остановленного spelled correctly? True
Is остоновочный spelled correctly? False
Correct options: ['остановочный', 'обстановочный', 'установочный', 'водосточный', 'стыковочный']
Word  остановленного normalized:  st:остановленный fl:A


#### Обработка

Проверка русских слов осуществляется в один этап -- с помощью spellchecker'а: если spellchecker подтверждает правильное написание слова, то слово существует в русском языке.

В силу сложной системы словоизменения в казахском языке проверка состоит из нескольких этапов:
1) проверка с помощью spellchecker'а
2) проверка в толковом словаре (на всякий случай)
3) итеративно убираем аффиксы с конца слова; сортируем каждую группу аффиксов по убыванию длины суффикса и сравниваем, оканчивается ли слово на этот суффикс; если оканчивается -- убираем его с конца слова и повторяем шаги 1) и 2) пока либо "обрезанное" слово не найдется в словаре или спелл-чекере, либо не останется морфем, которые можно удалить с конца

In [ ]:
import re
from string import punctuation

cyr_alph = 'АаБбВвГгДдЕеЁёЖжЗзИиЙйКкЛлМмНнОоПпРрСсТтУуФфХхЦцЧчШшЩщЪъЫыЬьЭэЮюЯя'
lat_alph = 'AaBbCcDdEeFfGgHhIiJjKkLlMmNnOoPpQqRrSsTtUuVvWwXxYyZz'
special_char = 'ӘәҒғҚқҢңӨөҰұҮүҺһІі'
alph = ''.join(set(list(cyr_alph + special_char + lat_alph)))

table = str.maketrans({
    'ə':'ә', # 601 -> 1241
    'Ə':'Ә', # 399 -> 1240
    'I':'І', # 73 -> 1030
    'i':'і' # 105 -> 1110
})

def normalize_special_chars(word):
    """
    Корректирует специализированные символы в казахских словах
    """
    return word.translate(table)

def extract_words(text, include_nonwords=True):
    """
    Извлекает из текста отдельные слова
    """
    text = re.sub(r'\\n', ' '*3, text)
    patt = f'[{alph}]+'

    tokens = []
    spans = []

    res = re.finditer(patt, text)
    for m in res:
        tokens.append(m.group())
        spans.append((m.start(0), m.end(0)))

    return tokens, spans

In [ ]:
df[['tokens', 'spans']] = df['text'].str.strip().apply(extract_words).apply(pd.Series)

In [ ]:
# noun_aff: [plural] [ы|і] [poss] [case] [adj] # https://aclanthology.org/W10-4124.pdf
noun_plural = ['лар', 'лер', 'дар', 'дер', 'тар', 'тер']
# noun_poss = ['ым', 'ім', 'м', 'ымыз', 'іміз', 'мыз', 'міз', 'ың', 'ің', 'ң', 'ыңыз', 'іңіз', 'ңыз', 'ңіз', 'сы', 'сі', 'ы', 'і']
noun_poss = ['м', 'мыз', 'міз', 'ң', 'ңыз', 'ңіз', 'сы', 'сі']
noun_case = ['дан', 'ден', 'тан', 'тен', 'нан', 'нен', 'да', 'де', 'та', 'те', 'нда', 'нде', 'ға', 'ге', 'қа', 'ке', 'на', 'не', 'а', 'е', 'дың', 'дің', 'тың', 'тің', 'ның', 'нің', 'ды', 'ді', 'ты', 'ті', 'ны', 'ні', 'н', 'бен', 'пен', 'мен']
noun_adj = ['ды', 'ді', 'ты', 'ті', 'лы', 'лі', 'сыз', 'сіз', 'ғы', 'гі', 'қы', 'кі', 'дай', 'дей', 'тай', 'тей', 'ндай', 'ндей', 'и', 'ы', 'дық', 'дік', 'тық', 'тік', 'лық', 'лік', 'паз', 'тал', 'ғыш', 'гіш', 'қыш', 'кіш', 'шек', 'шақ', 'шыл', 'шіл']

# verb_aff: ("ы"|"і") [voice] [neg] ["й"] [у] pers_{1,2} ["а"|"е"|"й"] tense [infl] [а|е|ы|и] [mood] [] # https://kaz-tili.kz/glag.htm
verb_voice = [
    'н', # возвр, страд
    'л', # страд
    'с', # совм
    'т', 'з', 'р', 'дыр', 'дір', 'тыр', 'тір', 'ғыз', 'гіз', 'қыз', 'кіз' # понуд
]
verb_neg = [
    'ба', 'бе', 'бы', 'бі',
    'па', 'пе', 'пы', 'пі',
    'ма', 'ме', 'мы', 'мі'
]
pers_1 = [
    'мын', 'мін', 'пын', 'пін', 'бын', 'бін',
    'мыз', 'міз', 'пыз', 'піз', 'быз', 'біз',
    'сың', 'сің',
    'сыңдар', 'сіңдер',
    'сыз', 'сіз',
    'сыздар', 'сіздер',
    'ды', 'ді', 'ты', 'ті'
]
pers_2 = [
    'м',
    'қ', 'к',
    'ң', 'ңдар', 'ңдер',
    'ңыз', 'ңіз',
    'ңыздар', 'ңіздер'
]

verb_pers = pers_1 + pers_2

verb_tense = [
    'уш', 'уші', # причастие наст. времени #ушы
    'ып', 'іп', 'п', # наст, давнопрош. неочев
    'д', 'ді', 'т', 'ті', # прошед #ды, ты
    'а', 'е', 'й', # переходн
    'са', 'се', # усл
    'ар', 'ер', 'р', # буд. предпол
    'бақ', 'бек', 'пақ', 'пек', 'мақ', 'мек', # буд. намер
    'ған', 'ген', 'қан', 'кен', # давнопрош. очев
    'атын', 'етін', 'йтын', 'йтін', # перех прош
    'ғал', 'гелі', 'қал', 'келі' # дееприч цели # ғалы, қалы
]
# verb_infl = ['ған', 'ген', 'қан', 'кен']
verb_mood_imper = [
    'йын', 'йін',
    'йық', 'йік',
    'ңдар', 'ңдер',
    'ңыз', 'ңіз',
    'ңыздар' ,'ңіздер',
    'сын', 'сін'
]
verb_mood_wish = [
    'ғы', 'гі', 'қы', 'кі'
]

verb_mood = verb_mood_imper + verb_mood_wish


In [ ]:
from collections import Counter
import re

def trim_affix(word, aff_list):
    # print(word)
    for aff in sorted(aff_list, key=lambda x: len(x))[::-1]:
        if word.endswith(aff):
            # print(word, aff)
            word = word[:-len(aff)]
            # print(word)
            return word
    return word

def stem_word(word):
    pos = [
        # verb
        # ("ы"|"i") [voice] [neg] [у] ["й"] pers_{1,2} ["а"|"е"|"й"] tense [infl] [а|е|ы|и] [mood]
        [verb_mood, ['а', 'е', 'ы', 'и'], verb_tense, ['а', 'е', 'й'], verb_pers, ['й'], ['у'], verb_neg, verb_voice, ['ы', 'і']],
        # noun
        # [plural] [ы|i] [poss] [case]
        [noun_adj, noun_case, noun_poss, ['ы', 'і'], noun_plural]
    ]

    for pos_seq in pos:
        for aff_lst in pos_seq:
            word = trim_affix(word, aff_lst)
            print(word)
    return word


def check_kaz(word_init):
    """
    Проверить слово или его производные в казахских словарях
    """

    def simple_lookup(t):
        return re.search(f'[{special_char}]', t) or kz_spellcheck.spell(t.title().encode('utf-8')) or t in kaz_dict # any(ch in 'ӘәҒғҚқҢңӨөҰұҮүҺһІі' for ch in t) or

    if simple_lookup(word_init):
        return True

    # если полное слово не нейдено в словаре, делаем стемминг
    pos = [
        # verb
        # ("ы"|"i") [voice] [neg] [у] ["й"] pers_{1,2} ["а"|"е"|"й"] tense [infl] [а|е|ы|и] [mood]
        [verb_mood, ['а', 'е', 'ы', 'и'], verb_tense, ['а', 'е', 'й'], verb_pers, ['й'], ['у'], verb_neg, verb_voice, ['ы', 'і']],
        # noun
        # [plural] [ы|i] [poss] [case]
        [noun_adj, noun_case, noun_poss, ['ы', 'і'], noun_plural]
    ]

    for pos_seq in pos:
        word = word_init
        for aff_lst in pos_seq:
            word = trim_affix(word, aff_lst)
            # print(word)
            if simple_lookup(word):
                return True
            # print(word)
    return False

def check_rus(word):
    """
    Проверить слово в русских словарях
    """
    return ru_spellcheck.spell(word.title().encode('utf-8')) # or word in rus_dict

def dict__sentence_level(tokens):
    """
    Проверяет каждое слово на наличие в казахском и русском языках. Если слово нашлось в обоих -- указывается тег ambig, если ни в одном -- то unk.

    Функция возвращает
    - язык, к которому принадлежит наибольшая доля слов
    - для каждого языка -- какая доля слов на этом языке используется в предложении
    - слово с указанием его порядкового номера и тег
    """

    c = Counter() # счетчик: сколько слов на каком языке
    res = {}

    for i, t in enumerate(tokens):
        lang = 'unk'

        if found.get(t):
            lang = found[t]

        else:
            if check_kaz(t):
                lang = 'kz'

            if check_rus(t):
                if lang=='unk':
                    lang = 'ru'
                else:
                    lang = 'ambig'

            found[t] = lang
        c.update([lang])

        res[f'{i}_{t}'] = lang

    perc = {x[0]: x[1]/c.total() for x in c.items()}

    # сортируем так, чтобы при равных вероятностях в качестве best_lang был 'kz' или'ru'
    dt = list(filter(lambda x: x[1] == c.most_common(1)[0][1] and x[0] in ('kz', 'ru'), list(c.most_common())))
    if dt:
        best_lang = dt[0][0]
    else:
        best_lang = c.most_common(1)[0][0]
    return best_lang, perc, res

In [ ]:
df_langproba = pd.DataFrame(index=exprmnt.df.index)

found = dict() # кэш
df_langproba['n_tokens'] = exprmnt.df['tokens'].apply(len)
df_langproba[[f'{METHOD}__best_lang', f'{METHOD}__probas', f'{METHOD}__token_annot']] = exprmnt.df['tokens'].apply(dict__sentence_level).tolist()
df_langproba = pd.concat([df_langproba, df_langproba[f'{METHOD}__probas'].apply(pd.Series)], axis=1)
df_langproba.rename(columns={
    'kz': f'{METHOD}__kaz',
    'ru': f'{METHOD}__rus',
    'ambig': f'{METHOD}__ambig',
    'unk': f'{METHOD}__unk'
}, inplace=True)

In [ ]:
print(*df_langproba['dict__token_annot'].to_list()[:10], sep='\n'+'='*80+'\n')

{'0_Сандж': 'unk', '1_тәуелсіз': 'kz', '2_зерттеу': 'kz', '3_орталығының': 'kz', '4_хабарлауынша': 'unk', '5_елімізде': 'kz', '6_коронавирус': 'kz', '7_жұқтырған': 'kz', '8_адам': 'ambig', '9_саны': 'ambig', '10_миллион': 'ambig', '11_адамға': 'kz', '12_жеткен': 'kz', '13_ал': 'kz', '14_ресми': 'kz', '15_дерек': 'kz', '16_бұдан': 'kz', '17_жүз': 'kz', '18_есе': 'kz', '19_аз': 'ambig', '20_Десек': 'kz', '21_те': 'ambig', '22_Тоқаев': 'kz', '23_коронавирусқа': 'kz', '24_қарсы': 'kz', '25_кезекті': 'kz', '26_жиын': 'kz', '27_өткізіп': 'kz', '28_жағдайды': 'kz', '29_көктемдегідей': 'kz', '30_жаппай': 'kz', '31_карантинге': 'kz', '32_жеткізбеуді': 'kz', '33_тапсырды': 'kz', '34_Әр': 'kz', '35_өңір': 'kz', '36_бойынша': 'kz', '37_карантин': 'ambig', '38_енгізіле': 'kz', '39_бастады': 'kz', '40_адамдар': 'kz', '41_тағы': 'kz', '42_да': 'ambig', '43_жұмысынан': 'kz', '44_айырылды': 'kz', '45_бірақ': 'kz', '46_теңге': 'kz', '47_төлем': 'kz', '48_осы': 'ambig', '49_жолы': 'kz', '50_болмайтын': '

In [ ]:
exprmnt.add_probas(df_langproba, dropped_cols=[f'{METHOD}__probas'])
exprmnt.df.head()

,text,manual_tag,cld2__all_langs,cld2__best_lang,cld2__kaz,cld2__rus,cld3__best_lang,cld3__kaz,cld3__rus,langid__best_lang,...,fasttext__rus,tokens,spans,n_tokens,dict__best_lang,dict__token_annot,dict__unk,dict__kaz,dict__ambig,dict__rus
0,“Сандж” тәуелсіз зерттеу орталығының хабарлауы...,kz,kz,kz,0.99,0.0,kk,0.999996,0.0,kk,...,0.00001,"[Сандж, тәуелсіз, зерттеу, орталығының, хабарл...","[(1, 6), (8, 16), (17, 24), (25, 36), (37, 49)...",76,kz,"{'0_Сандж': 'unk', '1_тәуелсіз': 'kz', '2_зерт...",0.039474,0.828947,0.131579,0.000000
1,Заңгер Алексей Сонның консультациясына мына сі...,kz,kz-unk,kz,0.98,0.0,kk,0.999802,0.0,kk,...,0.00001,"[Заңгер, Алексей, Сонның, консультациясына, мы...","[(0, 6), (7, 14), (15, 21), (22, 38), (39, 43)...",96,kz,"{'0_Заңгер': 'kz', '1_Алексей': 'ru', '2_Сонны...",0.041667,0.812500,0.125000,0.020833
2,Шымкентте елуге жуық әйел әкімдік алдына жинал...,kz,kz,kz,0.99,0.0,kk,0.999981,0.0,kk,...,0.00001,"[Шымкентте, елуге, жуық, әйел, әкімдік, алдына...","[(0, 9), (10, 15), (16, 20), (21, 25), (26, 33...",30,kz,"{'0_Шымкентте': 'unk', '1_елуге': 'kz', '2_жуы...",0.066667,0.933333,0.000000,0.000000
3,Өте маңызды жобаға сұхбат кейіпкері ретінде кі...,kz,kz,kz,0.98,0.0,kk,0.999996,0.0,kk,...,0.00001,"[Өте, маңызды, жобаға, сұхбат, кейіпкері, реті...","[(0, 3), (4, 11), (12, 18), (19, 25), (26, 35)...",40,kz,"{'0_Өте': 'kz', '1_маңызды': 'kz', '2_жобаға':...",0.025000,0.925000,0.050000,0.000000
4,Айсұлтанның “құпиясы” Британиялық Financial ti...,kz-unk,kz,kz,0.98,0.0,kk,0.999997,0.0,kk,...,0.00001,"[Айсұлтанның, құпиясы, Британиялық, Financial,...","[(0, 11), (13, 20), (22, 33), (34, 43), (44, 4...",106,kz,"{'0_Айсұлтанның': 'kz', '1_құпиясы': 'kz', '2_...",0.018868,0.886792,0.075472,0.018868


In [ ]:
exprmnt.df[f'{METHOD}__best_lang'].value_counts()

,count
dict__best_lang,
ru,5434
kz,3676
ambig,403
unk,186


In [ ]:
exprmnt.df[exprmnt.df[f'{METHOD}__best_lang']=='ambig'][f'{METHOD}__token_annot'].head().to_list()

[{'0_Алматы': 'kz',
  '1_площадь': 'ru',
  '2_Республики': 'ambig',
  '3_Фото': 'ambig'},
 {'0_Неге': 'ambig', '1_не': 'ambig', '2_емес': 'kz'},
 {'0_Морда': 'ambig',
  '1_как': 'ambig',
  '2_у': 'ambig',
  '3_спившегося': 'ru',
  '4_гитлера': 'ru'},
 {'0_Я': 'ru',
  '1_не': 'ambig',
  '2_таксист': 'ambig',
  '3_у': 'ambig',
  '4_меня': 'ru',
  '5_такая': 'ru',
  '6_позиция': 'ambig'},
 {'0_АКТ': 'ambig',
  '1_емес': 'kz',
  '2_акт': 'ambig',
  '3_Это': 'ru',
  '4_не': 'ambig',
  '5_акроним': 'ambig',
  '6_девочки': 'ru'}]

In [ ]:
exprmnt.df[[f'{METHOD}__kaz', f'{METHOD}__rus', f'{METHOD}__ambig', f'{METHOD}__unk']].describe()

,dict__kaz,dict__rus,dict__ambig,dict__unk
count,9699.000000,9699.000000,9699.000000,9699.000000
mean,0.336175,0.392021,0.202695,0.069109
std,0.408535,0.325684,0.150960,0.111178
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.083333,0.000000
50%,0.029412,0.500000,0.192308,0.019802
75%,0.828348,0.666667,0.300000,0.100000
max,1.000000,1.000000,1.000000,1.000000


In [ ]:
exprmnt.save_experiment(path=save_to_path)

In [ ]:
exprmnt.df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9699 entries, 0 to 9698
Data columns (total 38 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   text                        9699 non-null   object 
 1   manual_tag                  9699 non-null   object 
 2   cld2__all_langs             9699 non-null   object 
 3   cld2__best_lang             9699 non-null   object 
 4   cld2__kaz                   9699 non-null   float64
 5   cld2__rus                   9699 non-null   float64
 6   cld3__best_lang             9699 non-null   object 
 7   cld3__kaz                   9699 non-null   float64
 8   cld3__rus                   9699 non-null   float64
 9   langid__best_lang           9699 non-null   object 
 10  langid__kaz                 9699 non-null   float64
 11  langid__rus                 9699 non-null   float64
 12  lingua__best_lang           9699 non-null   object 
 13  lingua__kaz                 9699 